# Pneumonia Classifier — Colab (GPU) Training

**How to run:**
1. `Runtime` → `Change runtime type` → set **Hardware accelerator: GPU**.
2. `Runtime` → `Run all`.
3. When prompted, upload your Kaggle token `kaggle.json` — get it from kaggle.com → your profile → **Settings → API → Create New Token**.
4. The notebook downloads the dataset, trains on the GPU, and the **final cell downloads `model.pth`** to your computer.


In [ ]:
# --- Colab setup: install deps + fetch the dataset (no-op when not on Colab) ---
import os, sys

if "google.colab" in sys.modules:
    !pip -q install kaggle torchsummary

    # Upload your Kaggle API token once (kaggle.com -> Settings -> API -> Create New Token).
    from google.colab import files
    if not os.path.exists("/root/.kaggle/kaggle.json"):
        print("Upload kaggle.json:")
        files.upload()
        os.makedirs("/root/.kaggle", exist_ok=True)
        os.replace("kaggle.json", "/root/.kaggle/kaggle.json")
        os.chmod("/root/.kaggle/kaggle.json", 0o600)

    # Download + unzip the chest X-ray pneumonia dataset.
    if not os.path.exists("/content/data"):
        !kaggle datasets download -d paultimothymooney/chest-xray-pneumonia --unzip -p /content/data

    # The archive sometimes nests chest_xray/chest_xray -- locate the folder holding train/ and test/.
    for _root, _dirs, _ in os.walk("/content/data"):
        if "train" in _dirs and "test" in _dirs:
            os.environ["CHEST_XRAY_DIR"] = _root
            break
    print("CHEST_XRAY_DIR =", os.environ.get("CHEST_XRAY_DIR"))
else:
    print("Not on Colab; using CHEST_XRAY_DIR =", os.environ.get("CHEST_XRAY_DIR", "chest_xray"))


### libraries

In [ ]:
# Libraries
import os
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
import torchvision.transforms as transforms
import torchvision.models as models

from sklearn.metrics import classification_report, confusion_matrix

# Reproducibility: seed everything so runs are repeatable.
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

In [ ]:
from sklearn.model_selection import train_test_split


def create_dataset(folder_path):
    rows = []
    for category in ['NORMAL', 'PNEUMONIA']:  # the two classes
        category_path = os.path.join(folder_path, category)
        for file_name in os.listdir(category_path):
            file_path = os.path.join(category_path, file_name)
            if os.path.isfile(file_path) and file_name.lower().endswith(('.png', '.jpg', '.jpeg')):
                rows.append([file_path, category])
    return pd.DataFrame(rows, columns=['file_path', 'label'])


# Dataset location: defaults to ./chest_xray, override with the CHEST_XRAY_DIR env var.
# (No more hardcoded absolute path.)
dataset_dir = os.environ.get('CHEST_XRAY_DIR', 'chest_xray')
train_dir = os.path.join(dataset_dir, 'train')
test_dir = os.path.join(dataset_dir, 'test')

# Build dataframes and map labels: NORMAL -> 0, PNEUMONIA -> 1
full_train_df = create_dataset(train_dir)
test_df = create_dataset(test_dir)
full_train_df['label'] = full_train_df['label'].map({'NORMAL': 0, 'PNEUMONIA': 1})
test_df['label'] = test_df['label'].map({'NORMAL': 0, 'PNEUMONIA': 1})

# Carve a PROPER, stratified validation split from train (15%), seeded for reproducibility.
# This replaces the previous 20-image validation set, which was far too small to be meaningful,
# and it no longer mutates the dataset on disk.
train_df, val_df = train_test_split(
    full_train_df,
    test_size=0.15,
    stratify=full_train_df['label'],
    random_state=SEED,
)
train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)

print(f"Train set size: {len(train_df)}")
print(f"Validation set size: {len(val_df)}")
print(f"Test set size: {len(test_df)}")

In [ ]:
def count_categories(df, dataset_name):
    category_counts = df['label'].value_counts()
    print(f"{dataset_name} set:")
    print(f"  NORMAL: {category_counts.get(0, 0)}")
    print(f"  PNEUMONIA: {category_counts.get(1, 0)}")

# Count and display for train, validation, and test datasets
print("Image Counts per Category:")
count_categories(train_df, "Train")
count_categories(val_df, "Validation")
count_categories(test_df, "Test")

In [ ]:
def visualize_samples(df, n_samples=5):
    fig, axes = plt.subplots(2, n_samples, figsize=(15, 10))
    
    # Visualize 5 NORMAL samples
    normal_samples = df[df['label'] == 0].iloc[:n_samples]
    for i in range(n_samples):
        img_path = normal_samples.iloc[i, 0]
        label = normal_samples.iloc[i, 1]
        image = Image.open(img_path).convert('L')  # Open in grayscale (original format)
        axes[0, i].imshow(image, cmap='gray')
        axes[0, i].set_title(f"Label: {['Normal', 'Pneumonia'][label]}")
        axes[0, i].axis('off')
    
    # Visualize 5 PNEUMONIA samples
    pneumonia_samples = df[df['label'] == 1].iloc[:n_samples]
    for i in range(n_samples):
        img_path = pneumonia_samples.iloc[i, 0]
        label = pneumonia_samples.iloc[i, 1]
        image = Image.open(img_path).convert('L')  # Open in grayscale (original format)
        axes[1, i].imshow(image, cmap='gray')
        axes[1, i].set_title(f"Label: {['Normal', 'Pneumonia'][label]}")
        axes[1, i].axis('off')
        
    plt.show()

# Show 5 samples from the training data
visualize_samples(train_df)

In [ ]:
train_counts = train_df['label'].value_counts()
val_counts = val_df['label'].value_counts()
test_counts = test_df['label'].value_counts()

plt.figure(figsize=(8, 6))
plt.bar(['Pneumonia', 'Normal'], train_counts)
plt.title("Training Data Class Distribution")
plt.ylabel("Count")
plt.show()

plt.figure(figsize=(8, 6))
plt.bar(['Pneumonia', 'Normal'], val_counts)
plt.title("Validation Data Class Distribution")
plt.ylabel("Count")
plt.show()

plt.figure(figsize=(8, 6))
plt.bar(['Pneumonia', 'Normal'], test_counts)
plt.title("Test Data Class Distribution")
plt.ylabel("Count")
plt.show()

In [ ]:
from PIL import Image

# Custom dataset class
class ImageDataset(torch.utils.data.Dataset):
    def __init__(self, dataframe, transform=None):
        self.dataframe = dataframe
        self.transform = transform

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, idx):
        img_path = self.dataframe.iloc[idx, 0]  # Image path
        label = self.dataframe.iloc[idx, 1]     # Label (0 or 1)
        img = Image.open(img_path).convert('RGB')  # Convert to RGB if not already

        if self.transform:
            img = self.transform(img)

        return img, label

# Define transforms
train_transform_resnet = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.Grayscale(num_output_channels=3),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(5),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

val_transform_resnet = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.Grayscale(num_output_channels=3),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

test_transform_resnet = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.Grayscale(num_output_channels=3),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Create datasets
train_dataset_resnet = ImageDataset(train_df, transform=train_transform_resnet)
val_dataset_resnet = ImageDataset(val_df, transform=val_transform_resnet)
test_dataset_resnet = ImageDataset(test_df, transform=test_transform_resnet)

# Create DataLoaders.
# drop_last=True on training avoids a final batch of size 1, which would crash BatchNorm1d.
batch_size = 32
train_loader_resnet = DataLoader(train_dataset_resnet, batch_size=batch_size, shuffle=True, drop_last=True)
val_loader_resnet = DataLoader(val_dataset_resnet, batch_size=batch_size, shuffle=False)
test_loader_resnet = DataLoader(test_dataset_resnet, batch_size=batch_size, shuffle=False)

In [ ]:
from sklearn.utils.class_weight import compute_class_weight

# Class weights from the ACTUAL training split (handles the ~1:2.9 NORMAL:PNEUMONIA imbalance).
# Previously these were computed from hardcoded counts and then never used.
class_labels = np.array([0, 1])  # 0: NORMAL, 1: PNEUMONIA
class_weights = compute_class_weight('balanced', classes=class_labels, y=train_df['label'].values)
class_weights_tensor = torch.tensor(class_weights, dtype=torch.float)

print(f"Class weights: {dict(zip(class_labels, class_weights))}")

In [ ]:
import torch
from torch import nn, optim
from torchvision.models import resnet18, ResNet18_Weights

# Device setup
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Load ResNet-18 as a frozen feature extractor (transfer learning, not full fine-tuning).
model = resnet18(weights=ResNet18_Weights.DEFAULT)
for param in model.parameters():
    param.requires_grad = False  # Freeze the pretrained backbone

# New classifier head. IMPORTANT: ends in raw logits (NO Sigmoid/Softmax), because
# nn.CrossEntropyLoss applies log-softmax internally and expects logits.
in_features = model.fc.in_features
model.fc = nn.Sequential(
    nn.Linear(in_features, 128),
    nn.BatchNorm1d(128),
    nn.ReLU(),
    nn.Linear(128, 32),
    nn.BatchNorm1d(32),
    nn.ReLU(),
    nn.Linear(32, 2),  # 2 logits: NORMAL vs PNEUMONIA
)

model.to(device)

# Enable multi-GPU support if available
if torch.cuda.device_count() > 1:
    model = nn.DataParallel(model)

In [ ]:
import copy


def train_model(model, train_loader, val_loader, criterion, optimizer, num_epochs=10):
    """Train using train/validation only. The test set is NOT touched here
    (no per-epoch test evaluation = no selection leakage). The best model by
    validation loss is kept and restored at the end."""
    log_file = 'training_log.txt'
    train_losses, val_losses = [], []
    train_accuracies, val_accuracies = [], []
    best_val_loss = float('inf')
    best_state = copy.deepcopy(model.state_dict())

    with open(log_file, 'w') as log:
        log.write("Epoch, Train Loss, Train Accuracy, Val Loss, Val Accuracy\n")

        for epoch in range(num_epochs):
            # ---- Training ----
            model.train()
            running_loss, correct, total = 0.0, 0, 0
            for images, labels in train_loader:
                images, labels = images.to(device), labels.to(device)

                optimizer.zero_grad()
                outputs = model(images)
                loss = criterion(outputs, labels)
                loss.backward()
                optimizer.step()

                running_loss += loss.item()
                _, predicted = torch.max(outputs, 1)
                total += labels.size(0)
                correct += (predicted == labels).sum().item()

            train_loss = running_loss / len(train_loader)
            train_accuracy = 100 * correct / total
            train_losses.append(train_loss)
            train_accuracies.append(train_accuracy)

            # ---- Validation ----
            model.eval()
            val_loss, val_correct, val_total = 0.0, 0, 0
            with torch.no_grad():
                for images, labels in val_loader:
                    images, labels = images.to(device), labels.to(device)
                    outputs = model(images)
                    loss = criterion(outputs, labels)
                    val_loss += loss.item()
                    _, predicted = torch.max(outputs, 1)
                    val_total += labels.size(0)
                    val_correct += (predicted == labels).sum().item()

            val_loss /= len(val_loader)
            val_accuracy = 100 * val_correct / val_total
            val_losses.append(val_loss)
            val_accuracies.append(val_accuracy)

            # Track the best model by validation loss.
            if val_loss < best_val_loss:
                best_val_loss = val_loss
                best_state = copy.deepcopy(model.state_dict())

            print(f"Epoch [{epoch+1}/{num_epochs}] "
                  f"Train Loss: {train_loss:.4f}, Train Acc: {train_accuracy:.2f}% | "
                  f"Val Loss: {val_loss:.4f}, Val Acc: {val_accuracy:.2f}%")
            log.write(f"{epoch+1}, {train_loss:.4f}, {train_accuracy:.2f}, "
                      f"{val_loss:.4f}, {val_accuracy:.2f}\n")

    # Restore the best-validation weights before returning.
    model.load_state_dict(best_state)
    print(f"Best validation loss: {best_val_loss:.4f}")
    return train_losses, val_losses, train_accuracies, val_accuracies

In [ ]:
# Loss weighted for class imbalance; optimizer updates ONLY the trainable head params.
criterion = nn.CrossEntropyLoss(weight=class_weights_tensor.to(device))
optimizer = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=0.001)

# Train with validation-based model selection (test set untouched until final evaluation).
train_losses, val_losses, train_accuracies, val_accuracies = train_model(
    model, train_loader_resnet, val_loader_resnet, criterion, optimizer, num_epochs=10
)

In [ ]:
# Final evaluation on the held-out TEST set, run ONCE, using the best-validation model.
model.eval()
true_labels = []
predicted_labels = []

with torch.no_grad():
    for images, labels in test_loader_resnet:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        _, predicted = torch.max(outputs, 1)
        true_labels.extend(labels.cpu().numpy())
        predicted_labels.extend(predicted.cpu().numpy())

In [ ]:
# Classification report (precision / recall / F1 per class)
report = classification_report(true_labels, predicted_labels, target_names=['NORMAL', 'PNEUMONIA'])
print(report)

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

In [ ]:
# Compute and display the confusion matrix
cm = confusion_matrix(true_labels, predicted_labels)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['NORMAL', 'PNEUMONIA'])
disp.plot(cmap=plt.cm.YlGn)
plt.title("Confusion Matrix")
plt.show()

In [ ]:
# Accuracy per epoch (train vs validation)
plt.figure(figsize=(10, 5))
plt.plot(train_accuracies, label='Train Accuracy', color='blue', linestyle='--', marker='x')
plt.plot(val_accuracies, label='Validation Accuracy', color='red', linestyle='--', marker='x')
plt.title('ResNet-18 Accuracy per Epoch')
plt.xlabel('Epoch')
plt.ylabel('Accuracy (%)')
plt.legend()
plt.show()

In [ ]:
# Loss per epoch (train vs validation)
plt.figure(figsize=(10, 5))
plt.plot(train_losses, label='Train Loss', color='blue', linestyle='-', marker='o')
plt.plot(val_losses, label='Validation Loss', color='red', linestyle='-', marker='o')
plt.title('ResNet-18 Loss per Epoch')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
import matplotlib.pyplot as plt


def show_sample_predictions(model, test_loader, class_names, num_normal=10, num_pneumonia=10):
    """Display a balanced sample of test images (Normal + Pneumonia), with correct
    predictions titled in green and incorrect ones in red.

    Visual inspection only — the authoritative metrics are the classification report
    and confusion matrix above (computed over the full test set)."""
    model.eval()
    normal_images, pneumonia_images = [], []

    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            _, predictions = torch.max(model(images), 1)

            for i in range(images.size(0)):
                image = images[i].cpu().numpy().transpose(1, 2, 0)
                if image.shape[-1] == 1:  # grayscale
                    image = image.squeeze(-1)
                true_label, pred_label = labels[i].item(), predictions[i].item()
                sample = {
                    "image": image,
                    "true_label": class_names[true_label],
                    "pred_label": class_names[pred_label],
                    "correct": true_label == pred_label,
                }
                if true_label == 0 and len(normal_images) < num_normal:
                    normal_images.append(sample)
                elif true_label == 1 and len(pneumonia_images) < num_pneumonia:
                    pneumonia_images.append(sample)

            if len(normal_images) >= num_normal and len(pneumonia_images) >= num_pneumonia:
                break

    images_to_display = normal_images + pneumonia_images
    fig, axes = plt.subplots(4, 5, figsize=(15, 12))
    for idx, ax in enumerate(axes.flat):
        if idx < len(images_to_display):
            s = images_to_display[idx]
            ax.imshow(s["image"], cmap="gray" if s["image"].ndim == 2 else None)
            ax.set_title(f"True: {s['true_label']}\nPred: {s['pred_label']}",
                         color="green" if s["correct"] else "red")
        ax.axis("off")
    plt.tight_layout()
    plt.show()


print("Sample predictions (green = correct, red = wrong):")
show_sample_predictions(model, test_loader_resnet, class_names=["Normal", "Pneumonia"], num_normal=10, num_pneumonia=10)

In [ ]:
from torchsummary import summary

summary(model, input_size=(3, 224, 224))  # Adjust input_size based on your model's expected input

In [ ]:
# Save the best model (model currently holds the best-validation weights).
torch.save(model.state_dict(), 'model.pth')
print("Saved best model to model.pth")

In [ ]:
# Download the trained weights to your computer.
import os, sys

if "google.colab" in sys.modules:
    from google.colab import files
    files.download("model.pth")
else:
    print("model.pth saved at:", os.path.abspath("model.pth"))
